In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class EmotionNet(nn.Module):
    def __init__(self, num_classes=7):
        super(EmotionNet, self).__init__()
        # Khối 1: Conv -> ReLU -> Pool
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1) # 1 kênh vì ảnh xám
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
       
        # Khối 2
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
       
        # Lớp phân loại cuối cùng
        self.fc1 = nn.Linear(128 * 12 * 12, 512) # 48/2/2 = 12
        self.fc2 = nn.Linear(512, num_classes)
        self.dropout = nn.Dropout(0.5)


    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = F.relu(self.conv3(x))
       
        x = x.view(-1, 128 * 12 * 12) # Trải phẳng ma trận thành vector
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x


model = EmotionNet(num_classes=7)


In [4]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


# Chuẩn hóa ảnh: Chuyển về Tensor và đưa giá trị pixel về [0, 1]
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [5]:
# Dataset cho huấn luyện
train_dataset = datasets.ImageFolder(
    root=r'E:\projectNMAI\model\dataset\train', # Trỏ vào thư mục train
    transform=transform
)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Dataset cho kiểm tra (Test)
test_dataset = datasets.ImageFolder(
    root=r'E:\projectNMAI\model\dataset\test', # Trỏ vào thư mục test
    transform=transform
)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Đã tải {len(train_dataset)} ảnh train và {len(test_dataset)} ảnh test.")

Đã tải 28709 ảnh train và 6142 ảnh test.


In [6]:
import torch.optim as optim


criterion = nn.CrossEntropyLoss() # Dùng cho phân loại nhiều lớp
optimizer = optim.Adam(model.parameters(), lr=0.001)


# Chuyển mô hình sang GPU nếu có
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

EmotionNet(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=18432, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=7, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

In [7]:
import os
from PIL import Image

# Đường dẫn đến tập dữ liệu bạn đang dùng để train
test_dir = r'E:\projectNMAI\model\dataset\test'

print("🧼 Đang dọn dẹp tập dữ liệu...")
count = 0
for root, dirs, files in os.walk(test_dir):
    for file in files:
        file_path = os.path.join(root, file)
        try:
            with Image.open(file_path) as img:
                img.verify() # Kiểm tra tính toàn vẹn của ảnh
        except Exception:
            print(f"🗑️ Đã xóa file lỗi: {file_path}")
            os.remove(file_path)
            count += 1

print(f"✨ Hoàn tất! Đã loại bỏ {count} file lỗi.")

🧼 Đang dọn dẹp tập dữ liệu...
✨ Hoàn tất! Đã loại bỏ 0 file lỗi.


In [7]:
epochs = 20
model.to(device) # Đảm bảo model đã nằm trên GPU/CPU

for epoch in range(epochs):
    # --- GIAI ĐOẠN HUẤN LUYỆN (TRAINING) ---
    model.train() 
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    # --- GIAI ĐOẠN KIỂM TRA (TESTING/VALIDATION) ---
    model.eval() # Chuyển sang chế độ đánh giá (tắt Dropout)
    correct = 0
    total = 0
    with torch.no_grad(): # Tắt tính toán đạo hàm để tiết kiệm RAM/GPU
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1) # Lấy lớp có xác suất cao nhất
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    epoch_loss = running_loss / len(train_loader)
    
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f} - Accuracy: {accuracy:.2f}%")

Epoch [1/20] - Loss: 1.5896 - Accuracy: 43.49%
Epoch [2/20] - Loss: 1.3290 - Accuracy: 48.26%
Epoch [3/20] - Loss: 1.1763 - Accuracy: 52.57%
Epoch [4/20] - Loss: 1.0349 - Accuracy: 54.56%
Epoch [5/20] - Loss: 0.8766 - Accuracy: 55.85%
Epoch [6/20] - Loss: 0.7181 - Accuracy: 56.30%
Epoch [7/20] - Loss: 0.5704 - Accuracy: 55.83%
Epoch [8/20] - Loss: 0.4428 - Accuracy: 56.94%
Epoch [9/20] - Loss: 0.3458 - Accuracy: 56.79%
Epoch [10/20] - Loss: 0.2931 - Accuracy: 56.12%
Epoch [11/20] - Loss: 0.2430 - Accuracy: 55.65%
Epoch [12/20] - Loss: 0.2172 - Accuracy: 56.06%
Epoch [13/20] - Loss: 0.1960 - Accuracy: 57.05%
Epoch [14/20] - Loss: 0.1708 - Accuracy: 55.88%
Epoch [15/20] - Loss: 0.1663 - Accuracy: 55.99%
Epoch [16/20] - Loss: 0.1519 - Accuracy: 55.93%
Epoch [17/20] - Loss: 0.1458 - Accuracy: 55.98%
Epoch [18/20] - Loss: 0.1377 - Accuracy: 56.19%
Epoch [19/20] - Loss: 0.1356 - Accuracy: 56.45%
Epoch [20/20] - Loss: 0.1292 - Accuracy: 56.24%
